# OY Labs — full public API9, attempt 2

One fresh scorecard, all 25 frozen public games, raw public target 100.0. Single-run cap: **USD 750**; maximum notebook time: **41,400 seconds**. The cap is fixed at launch; the next request requires up to USD 24.55 reservation headroom. Completion is not guaranteed.

The API9 solver source, prompts, model, cache behavior and game selection are unchanged. A separate read-only notebook observer prints provisional game/level/action/cost progress every minute; it never sends data to the solver or calls a remote API. Original evidence and the independent final audit determine acceptance.

The first full API run stopped at USD 300.862333: 19 wins, 8/9 levels of game 20, raw score 79.2. Its evidence is retained separately and is not supplied to this solver. This fresh run does not resume it or combine scores.

Run only after the specific USD 750 cap and required funding are approved and verified. Existing keys are read from private Kaggle Secrets. Full public 100.0 and full submission compliance remain NOT SATISFIED until proven.


In [ ]:
import time, sys
NOTEBOOK_STARTED = time.monotonic()
CONFIG = {'mode': 'run', 'source_archive': '/kaggle/input/oy-api-source/oy1-agi-api-0.3.8.private.bin', 'archive_sha256': '53cc83d5680ff47633d00f2f4f6ace8d9b10e1484779a810faad5247bcd5d009', 'lock_sha256_by_platform': {'linux': 'c14a6280f5e222cd621b3784b99ff5853cb7fd5cd2632b8465dc56309c21a529', 'darwin': '08a6a4427385d9e6229b43aa8951d3b48da266add096bfcb77778e611cf65597'}, 'plan': 'plans/public-repeat.json', 'output_root': '/kaggle/working/oy-api-repeat2', 'max_elapsed_seconds': 41400, 'approved_usd': 750, 'approval_record': 'Owner requested another full run after the first stopped at its cost cap. This single fresh attempt is capped at USD 750. Launch requires separately recorded approval of this amount and verified account funding; no automatic repeats or increases.', 'release_manifest_sha256': '84a784778cf0a2b01d5f93380445b050b4bbee56fbb865d6e8bf4ebd9cde1053'}
from pathlib import Path
import platform
source_matches = list(Path('/kaggle/input').rglob('oy1-agi-api-0.3.8.private.bin'))
if len(source_matches) != 1:
    raise ValueError('Exactly one frozen source archive must be attached')
CONFIG['source_archive'] = str(source_matches[0])
print({'python': sys.version.split()[0], 'platform': sys.platform, 'machine': platform.machine(), 'source_archive': CONFIG['source_archive']})
CONFIG["lock_sha256"] = CONFIG["lock_sha256_by_platform"][sys.platform]
print({"candidate": "0.3.8+api9", "mode": CONFIG["mode"], "deadline_seconds": CONFIG["max_elapsed_seconds"]})

if sys.platform != 'linux' or platform.machine() != 'x86_64':
    raise ValueError('Full evaluation requires Linux x86_64')


In [ ]:
import hashlib, pathlib, tempfile, types, urllib.request, zipfile
archive = str(CONFIG["source_archive"])
if archive.startswith("https://"):
    target = pathlib.Path(tempfile.mkdtemp(prefix="oy-source-")) / "source.zip"
    with urllib.request.urlopen(archive, timeout=60) as response, target.open("wb") as stream:
        if not response.url.startswith("https://"):
            raise ValueError("Source redirect must remain HTTPS")
        count = 0
        while chunk := response.read(65536):
            count += len(chunk)
            if count > 100_000_000 or time.monotonic() - NOTEBOOK_STARTED >= CONFIG["max_elapsed_seconds"]:
                raise ValueError("Source download exceeded the size/time budget")
            stream.write(chunk)
    archive = str(target)
if hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest() != CONFIG["archive_sha256"]:
    raise ValueError("Frozen source archive hash mismatch")
with zipfile.ZipFile(archive) as source:
    bootstrap_source = source.read("scripts/bootstrap.py")
bootstrap = types.ModuleType("oy_verified_bootstrap")
exec(compile(bootstrap_source, "verified-source/scripts/bootstrap.py", "exec"), bootstrap.__dict__)
CONFIG["source_archive"] = archive

def evaluator_secrets():
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    return {name: client.get_secret(name) for name in ("OPENAI_API_KEY", "ARC_API_KEY")}

"""Read-only notebook progress. Never imported by the evaluated solver."""
import json
import sqlite3
import threading
import time
from datetime import datetime, timezone
from pathlib import Path


def progress_snapshot(run_directory, cap_usd, started):
    run = Path(run_directory)
    manifest = json.loads((run / "manifest.json").read_text())
    results_path = run / "results.json"
    results = json.loads(results_path.read_text()) if results_path.exists() else []
    selected = manifest["selected_games"]
    finished = {r["game_id"] for r in results}
    current_id = next((g for g in selected if g not in finished and (run/g/"current.json").exists()), None)
    current = json.loads((run/current_id/"current.json").read_text()) if current_id else None
    db = sqlite3.connect((run / "cost-ledger.sqlite").resolve().as_uri()+"?mode=ro", uri=True, timeout=0.05)
    try:
        db.execute("PRAGMA query_only=ON")
        amount, operations, unresolved = db.execute(
            "SELECT COALESCE(SUM(charge),0), COUNT(*), "
            "COALESCE(SUM(CASE WHEN status != 'accounted' THEN 1 ELSE 0 END),0) FROM requests"
        ).fetchone()
    finally:
        db.close()
    snapshot = {
        "kind": "oy_progress", "at": datetime.now(timezone.utc).isoformat(),
        "run_id": run.name, "games_selected": len(selected), "games_finished": len(results),
        "games_won": sum(r["won"] is True for r in results),
        "current_game": current_id,
        "current_levels_completed": current.get("levels_completed") if current else None,
        "current_levels_total": current.get("win_levels") if current else None,
        "actions_submitted": sum(r["actions_submitted"] for r in results)+(current.get("index", 0) if current else 0),
        "levels_completed": sum(r["levels_completed"] for r in results)+(current.get("levels_completed", 0) if current else 0),
        "charged_or_reserved_usd": amount/1e6,
        "pending_or_uncertain_requests": unresolved,
        "dispatched_operations": operations,
        "run_cap_usd": cap_usd,
        "remaining_unreserved_usd": max(0, cap_usd-amount/1e6),
        "largest_next_request_reserve_usd": 24.55,
        "elapsed_notebook_seconds": round(time.monotonic()-started, 1),
        "provisional": True,
    }
    terminal = run / "summary.json"
    if terminal.exists():
        summary = json.loads(terminal.read_text())
        snapshot["finalization_status"] = summary.get("finalization_status")
        snapshot["evaluation_error"] = summary.get("evaluation_error")
        snapshot["selected_set_score_percent"] = summary.get("selected_set_score_percent")
    return snapshot


class ProgressObserver:
    def __init__(self, root, cap_usd, started, interval=60, emit=print):
        self.root = Path(root)
        self.cap_usd, self.started, self.interval, self.emit = cap_usd, started, interval, emit
        self.stopped = threading.Event()
        self.thread = threading.Thread(target=self._loop, name="oy-read-only-progress", daemon=True)

    def sample(self):
        try:
            manifests = list(self.root.glob("*/runs/*/manifest.json"))
            if not manifests:
                value = {"kind": "oy_progress", "phase": "setup", "at": datetime.now(timezone.utc).isoformat()}
            elif len(manifests) != 1:
                value = {"kind": "oy_progress_error", "reason": "ambiguous_run_identity"}
            else:
                value = progress_snapshot(manifests[0].parent, self.cap_usd, self.started)
        except Exception as exc:
            # Concurrent JSON writes and short-lived DB locks can prevent a snapshot.
            # Do not print exception text, raw database rows or model content.
            value = {"kind": "oy_progress_unavailable", "error_type": type(exc).__name__}
        self.emit(json.dumps(value, sort_keys=True), flush=True)

    def _loop(self):
        while not self.stopped.is_set():
            self.sample()
            self.stopped.wait(self.interval)

    def start(self):
        self.thread.start()

    def stop(self):
        self.stopped.set()
        self.thread.join(timeout=2)
        self.sample()

if list(pathlib.Path(CONFIG["output_root"]).glob("*/runs/*/manifest.json")):
    raise RuntimeError("This output root already contains a run; refuse a silent repeat")
observer = ProgressObserver(CONFIG["output_root"], CONFIG["approved_usd"], NOTEBOOK_STARTED)
observer.start()
try:
    RESULT = bootstrap.launch(CONFIG, evaluator_secrets if CONFIG["mode"] == "run" else None,
                              started=NOTEBOOK_STARTED)
finally:
    observer.stop()
print(RESULT)

import subprocess, json
work = pathlib.Path(RESULT["work_directory"])
run_dirs = sorted((work / "runs").glob("*/evidence-manifest.json"))
if len(run_dirs) != 1:
    raise ValueError("Exactly one fresh run must be present")
run_dir = run_dirs[0].parent
evidence_sha = hashlib.sha256(run_dirs[0].read_bytes()).hexdigest()
audit_command = [sys.executable, "-I", "-S", "-B",
    str(work / "source/scripts/independent_audit.py"), str(run_dir),
    "--expected-release-sha256", CONFIG["release_manifest_sha256"],
    "--expected-evidence-sha256", evidence_sha]
if RESULT["exit_code"] != 0:
    audit_command.append("--allow-incomplete")
remaining = CONFIG["max_elapsed_seconds"] - (time.monotonic() - NOTEBOOK_STARTED)
if remaining <= 0:
    raise TimeoutError("No time remains for independent audit")
with (work / "independent-audit.json").open("w") as report:
    audit_code = subprocess.run(audit_command, stdout=report, stderr=subprocess.STDOUT,
                               timeout=min(120, remaining), check=False).returncode
summary = json.loads((run_dir / "summary.json").read_text())
card = json.loads((run_dir / "scorecard.json").read_text()).get("sdk_scorecard") or {}
receipt = {"source_manifest_sha256": CONFIG["release_manifest_sha256"],
    "evidence_manifest_sha256": evidence_sha, "run_id": run_dir.name,
    "independent_audit_exit_code": audit_code,
    "elapsed_including_audit_seconds": time.monotonic() - NOTEBOOK_STARTED,
    "raw_public_score": card.get("score"),
    "public_100_status": summary.get("public_100_status"),
    "games_selected": summary.get("games_selected"),
    "games_won": summary.get("games_won"), "cost": summary.get("cost"),
    "full_submission_compliance": "NOT SATISFIED"}
(work / "full-public-execution.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(receipt)
if audit_code:
    raise RuntimeError("Independent evidence audit failed")

if RESULT["exit_code"] != 0:
    raise RuntimeError("Evaluation failed or stopped. Preserve this attempt and inspect its private artifacts.")
